In [30]:
import pandas as pd
import feature_engineering_helper as hf
import importlib
from sklearn.model_selection import StratifiedShuffleSplit

In [31]:
# get raw data
data_prefix = '../data_curation/processed_data/'
original_train = pd.read_csv(data_prefix + 'train_without_data_augmentation.csv')[['SMILES','MP']]

original_test = pd.read_csv(data_prefix + 'test_predictions.csv')[['SMILES','exp MP']]
original_test = original_test.rename(columns={'exp MP': 'MP'})

raw_data = pd.concat([original_train, original_test], axis=0).reset_index(drop=True)

print(original_train.shape)
print(original_test.shape)
print(raw_data.shape)

(17633, 2)
(1961, 2)
(19594, 2)


In [32]:
# Clean data with new melting point bounds (between low_bound and high_bound)
clean_data = hf.clean_dataset(raw_data, low_bound=0, high_bound=500)

Initial shape: (19594, 2)
Removed 2370 rows with MP < 0
Removed 2 rows with MP > 500
Final shape: (17222, 2)


In [33]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import numpy as np

def compute_mw(df, smiles_col="SMILES"):
    mw_list = []
    
    for smi in df[smiles_col]:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            mw_list.append(Descriptors.MolWt(mol))
        else:
            mw_list.append(np.nan)
    
    df = df.copy()
    df["MW"] = mw_list
    return df


def compute_mw_quantile_thresholds(df, quantiles=[0.6, 0.7, 0.8, 0.9]):
    
    print("\nMW Quantile Thresholds:")
    thresholds = {}
    
    for q in quantiles:
        threshold = df["MW"].quantile(q)
        thresholds[q] = threshold
        print(f"Quantile {q:.2f} → MW threshold = {threshold:.2f}")
    
    return thresholds


def add_mw_label_by_quantile(df, quantile=0.5):
    
    threshold = df["MW"].quantile(quantile)
    
    df = df.copy()
    df["MW_label"] = (df["MW"] > threshold).astype(int)
    
    print(f"\nUsing quantile {quantile}")
    print(f"Threshold = {threshold:.2f}")
    print(df["MW_label"].value_counts())
    
    return df, threshold

In [34]:
clean_data = compute_mw(clean_data)
thresholds = compute_mw_quantile_thresholds(clean_data)

clean_data, threshold = add_mw_label_by_quantile(clean_data, quantile=0.9)
clean_data.describe()



MW Quantile Thresholds:
Quantile 0.60 → MW threshold = 254.25
Quantile 0.70 → MW threshold = 286.33
Quantile 0.80 → MW threshold = 326.30
Quantile 0.90 → MW threshold = 384.52

Using quantile 0.9
Threshold = 384.52
MW_label
0    15499
1     1723
Name: count, dtype: int64


,MP,MW,MW_label
count,17222.000000,17222.000000,17222.000000
mean,126.045995,250.758289,0.100046
std,70.913439,106.137789,0.300071
min,0.000000,18.015000,0.000000
25%,69.000000,176.137000,0.000000
50%,120.000000,227.169000,0.000000
75%,175.000000,305.205000,0.000000
max,492.500000,1701.206000,1.000000


In [35]:
data_with_features = hf.smiles_to_features(clean_data, ['rdkit'])
data_with_features

✓ RDKit: Added 217 features


/Users/sdl5_mp/Documents/GitHub/melting_point_2026/MW/feature_engineering_helper.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_with_features[f'RDKit_{name}'] = [f[i] for f in rdkit_features]
/Users/sdl5_mp/Documents/GitHub/melting_point_2026/MW/feature_engineering_helper.py:43: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  data_with_features[f'RDKit_{name}'] = [f[i] for f in rdkit_features]
/Users/sdl5_mp/Documents/GitHub/melting_point_2026/MW/feature_engineering_helper.py:43: PerformanceWarning: DataFrame is highly 

,MP,MW,MW_label,RDKit_AvgIpc,RDKit_BCUT2D_CHGHI,RDKit_BCUT2D_CHGLO,RDKit_BCUT2D_LOGPHI,RDKit_BCUT2D_LOGPLOW,RDKit_BCUT2D_MRHI,RDKit_BCUT2D_MRLOW,...,RDKit_fr_sulfone,RDKit_fr_term_acetylene,RDKit_fr_tetrazole,RDKit_fr_thiazole,RDKit_fr_thiocyan,RDKit_fr_thiophene,RDKit_fr_unbrch_alkane,RDKit_fr_urea,RDKit_qed,SMILES
0,0.0,172.180,0,2.021826,1.969209,-1.950422,1.839529,-2.030578,5.910992,-0.139527,...,0,0,0,0,0,0,0,0,0.461644,CCOC(=O)/C=C/C(=O)OCC
1,230.0,456.108,1,2.596470,2.134805,-2.073420,2.259390,-2.109867,6.349561,0.072022,...,0,0,0,0,0,0,0,0,0.322574,O=C(c1ccc(cc1)C(=O)Oc1cc(Cl)cc(c1)Cl)Oc1cc(Cl)...
2,285.0,420.472,1,2.841923,2.064669,-2.060442,2.237060,-2.108735,6.045732,0.101400,...,0,0,0,0,0,0,0,0,0.344216,O=C(c1ccccc1)Nc1cccc(c1)/N=N/c1cccc(c1)NC(=O)c...
4,112.0,139.154,0,2.046001,1.922596,-1.962809,1.975790,-1.933211,5.089489,0.265376,...,0,0,0,0,0,0,0,0,0.609080,OCc1cccc(n1)CO
5,160.0,344.827,0,3.160349,2.275297,-2.119670,2.398983,-2.132280,7.992453,0.414605,...,0,0,0,0,0,0,0,0,0.779142,COc1ccc(cc1)c1nnc2n1NC(S2)c1ccc(cc1)Cl
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19588,49.0,172.130,0,1.956738,2.043838,-1.965025,2.082921,-2.002395,5.695900,-0.135729,...,0,0,0,0,0,0,0,0,0.735003,OC(=O)Cc1ccc(c(c1)F)F
19589,72.0,241.290,0,2.416826,2.177119,-2.214041,2.250524,-2.260984,5.953675,0.162449,...,0,0,0,0,0,0,0,0,0.812970,CCOC(=O)N(c1ccccc1)c1ccccc1
19591,90.0,205.040,0,1.956738,2.023046,-1.983385,2.220807,-1.989227,6.415186,-0.135725,...,0,0,0,0,0,0,0,0,0.805020,OC(=O)Cc1ccc(c(c1)Cl)Cl
19592,148.0,212.201,0,2.213081,2.134795,-2.053354,2.349259,-2.050401,5.905731,0.050385,...,0,0,0,0,0,0,0,0,0.519957,CCCOC(=O)c1cc(O)c(c(c1)O)O


In [36]:
# drop rows with NaN values and print how many rows were dropped and reset index
nan_rows = data_with_features[data_with_features.isna().any(axis=1)]
num_dropped = nan_rows.shape[0]
print(f"Dropped {num_dropped} rows containing NaN values.")
data_with_features = data_with_features.dropna().reset_index(drop=True)

Dropped 2 rows containing NaN values.


In [37]:
# stratified split based on  compliance

split = StratifiedShuffleSplit(n_splits=1, test_size=0.4, random_state=42)
for train_index, test_index in split.split(data_with_features, data_with_features['MW_label']):
    strat_train_set = data_with_features.loc[train_index]
    strat_test_set = data_with_features.loc[test_index]
    strat_train_set['Type'] = 'Train'
    strat_test_set['Type'] = 'Test'
final_data = pd.concat([strat_train_set, strat_test_set], axis=0).reset_index(drop=True)

# print the shape of train and test dataset
print(f"Train set: {final_data[final_data['Type'] == 'Train'].shape}")
print(f"Test set: {final_data[final_data['Type'] == 'Test'].shape}")

Train set: (10332, 222)
Test set: (6888, 222)


In [38]:
data_prefix_MW = '../MW/artifacts_threshold/'

final_data.to_parquet(data_prefix_MW + 'all_features_RDKit_60(split)_MW(label)_90(threshold).parquet', compression= "zstd")